In [1]:
import pandas as pd
import numpy as np

In [2]:
def combine_stats(df):
    df = df.copy()
    df.columns = df.columns.str.upper()
    df['W1COLUMN'] = df['WSCORE']
    df['W2COLUMN'] = df['LSCORE']

    # Winning rows: SEASON, DAYNUM, all W-prefixed cols, WON=1
    w_cols = ['SEASON', 'DAYNUM'] + [c for c in df.columns if c.startswith('W')]
    winning = df[w_cols].copy()
    winning['WON'] = 1
    rename_map = {
        c: c[1:] for c in winning.columns
        if c.startswith('W') and c not in ['WON', 'WLOC', 'W1COLUMN', 'W2COLUMN']
    }
    winning = winning.rename(columns=rename_map)

    # Losing rows: SEASON, DAYNUM, WLOC, W1COLUMN, W2COLUMN, all L-prefixed cols, WON=0
    l_cols = ['SEASON', 'DAYNUM', 'WLOC', 'W1COLUMN', 'W2COLUMN'] + [
        c for c in df.columns if c.startswith('L')
    ]
    losing = df[l_cols].copy()
    losing['WON'] = 0
    rename_map = {
        c: c[1:] for c in losing.columns
        if c.startswith('L') and c not in ['WON', 'WLOC']
    }
    losing = losing.rename(columns=rename_map)

    return pd.concat([winning, losing], ignore_index=True)

In [3]:
m_reg = combine_stats(pd.read_csv('data_2026/WRegularSeasonDetailedResults.csv'))

In [4]:
m_reg

,SEASON,DAYNUM,TEAMID,SCORE,WLOC,FGM,FGA,FGM3,FGA3,FTM,...,OR,DR,AST,TO,STL,BLK,PF,W1COLUMN,W2COLUMN,WON
0,2010,11,3103,63,H,23,54,5,9,12,...,10,26,14,18,7,0,15,63,49,1
1,2010,11,3104,73,N,26,62,5,12,16,...,16,31,15,20,5,2,25,73,68,1
2,2010,11,3110,71,A,29,62,6,15,7,...,14,23,18,13,6,2,17,71,59,1
3,2010,11,3111,63,A,27,52,4,11,5,...,6,40,14,27,5,10,18,63,58,1
4,2010,11,3119,74,H,30,74,7,20,7,...,14,33,18,11,5,3,18,74,70,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174369,2026,131,3218,48,N,18,48,4,16,8,...,2,31,9,17,3,6,15,60,48,0
174370,2026,132,3220,56,N,23,58,7,21,3,...,9,25,10,13,2,4,17,68,56,0
174371,2026,132,3254,57,H,19,55,7,20,12,...,5,19,2,11,5,4,15,79,57,0
174372,2026,132,3250,70,H,26,55,9,24,9,...,8,17,17,11,6,2,12,77,70,0


In [5]:
w_margin = (
    m_reg[m_reg.WON == 1]
    .assign(SCOREDIFF=lambda x: x.W1COLUMN - x.W2COLUMN)
    .groupby(['SEASON', 'TEAMID'])
    .agg(WINMARGINMEDIAN=('SCOREDIFF', 'median'), WINMARGINMEAN=('SCOREDIFF', 'mean'))
    .reset_index()
)

l_margin = (
    m_reg[m_reg.WON == 0]
    .assign(SCOREDIFF=lambda x: x.W1COLUMN - x.W2COLUMN)
    .groupby(['SEASON', 'TEAMID'])
    .agg(LOSEMARGINMEDIAN=('SCOREDIFF', 'median'), LOSEMARGINMEAN=('SCOREDIFF', 'mean'))
    .reset_index()
)

m_season_margin = w_margin.merge(l_margin, on=['SEASON', 'TEAMID'])
m_reg = m_reg.drop(columns=['W1COLUMN', 'W2COLUMN'])

In [6]:
num_cols = [
    c for c in m_reg.drop(columns=['DAYNUM', 'SEASON', 'TEAMID', 'WLOC']).columns
    if m_reg[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]

agg_dict = {}
for col in num_cols:
    agg_dict[f'{col}_MEAN'] = (col, 'mean')
    agg_dict[f'{col}_MEDIAN'] = (col, 'median')
    agg_dict[f'{col}_STDDEV'] = (col, 'std')

season_stats = (
    m_reg.drop(columns=['DAYNUM'])
    .groupby(['SEASON', 'TEAMID'])
    .agg(**agg_dict)
    .reset_index()
)

# Drop aggregated stat columns for WON, SEASON, TEAMID (not meaningful as averages)
cols_to_drop = [
    c for c in season_stats.columns
    if c.startswith('WON_') or c.startswith('SEASON_') or c.startswith('TEAMID_')
]
season_stats = season_stats.drop(columns=cols_to_drop)

In [7]:
hna = (
    m_reg.groupby(['SEASON', 'TEAMID', 'WLOC'])
    .agg(WINCOUNT=('WON', 'sum'))
    .reset_index()
)
hna['WLOC'] = 'WLOC' + hna['WLOC']
hna = (
    hna.pivot_table(index=['SEASON', 'TEAMID'], columns='WLOC', values='WINCOUNT', fill_value=0)
    .reset_index()
)
hna.columns.name = None

In [8]:
season_joined = (
    season_stats
    .merge(hna, on=['SEASON', 'TEAMID'])
    .merge(m_season_margin, on=['SEASON', 'TEAMID'])
    .drop_duplicates()
    .fillna(0)
)

In [9]:
season_joined[(season_joined.SEASON == 2021) & (season_joined.TEAMID == 1288)]

,SEASON,TEAMID,SCORE_MEAN,SCORE_MEDIAN,SCORE_STDDEV,FGM_MEAN,FGM_MEDIAN,FGM_STDDEV,FGA_MEAN,FGA_MEDIAN,...,PF_MEAN,PF_MEDIAN,PF_STDDEV,WLOCA,WLOCH,WLOCN,WINMARGINMEDIAN,WINMARGINMEAN,LOSEMARGINMEDIAN,LOSEMARGINMEAN


In [10]:
conf_games = pd.read_csv('data_2026/WConferenceTourneyGames.csv')
conf_games.columns = conf_games.columns.str.upper()

# Get the conference champion (last game = highest DAYNUM per SEASON, CONFABBREV)
conf_wins = (
    conf_games
    .sort_values('DAYNUM', ascending=False)
    .groupby(['SEASON', 'CONFABBREV'], as_index=False)
    .first()
    .drop(columns=['DAYNUM', 'LTEAMID', 'CONFABBREV'])
    .assign(WON_CONFERENCE=1)
    .rename(columns={'WTEAMID': 'TEAMID'})
)

final = season_joined.merge(conf_wins, on=['SEASON', 'TEAMID'], how='left')
final['WON_CONFERENCE'] = final['WON_CONFERENCE'].fillna(0)
final['TOTAL_WINS'] = final['WLOCN'] + final['WLOCH'] + final['WLOCA']

In [11]:
final[(final.SEASON == 2024) & (final.TEAMID == 1314)]

,SEASON,TEAMID,SCORE_MEAN,SCORE_MEDIAN,SCORE_STDDEV,FGM_MEAN,FGM_MEDIAN,FGM_STDDEV,FGA_MEAN,FGA_MEDIAN,...,PF_STDDEV,WLOCA,WLOCH,WLOCN,WINMARGINMEDIAN,WINMARGINMEAN,LOSEMARGINMEDIAN,LOSEMARGINMEAN,WON_CONFERENCE,TOTAL_WINS


In [12]:
# season is the materialized season stats dataframe (same as final at this point)
season = final.copy()

In [13]:
seeds = pd.read_csv('data_2026/WNCAATourneySeeds.csv')
seeds.columns = seeds.columns.str.upper()
seeds

,SEASON,SEED,TEAMID
0,1998,W01,3330
1,1998,W02,3163
2,1998,W03,3112
3,1998,W04,3301
4,1998,W05,3272
...,...,...,...
1807,2026,Z12,3211
1808,2026,Z13,3453
1809,2026,Z14,3158
1810,2026,Z15,3239


In [14]:
seeds['REGION'] = seeds['SEED'].str[0]
seeds['SEED'] = seeds['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)
seed_value = seeds[['SEASON', 'TEAMID', 'REGION', 'SEED']].copy()
seed_value

,SEASON,TEAMID,REGION,SEED
0,1998,3330,W,1
1,1998,3163,W,2
2,1998,3112,W,3
3,1998,3301,W,4
4,1998,3272,W,5
...,...,...,...,...
1807,2026,3211,Z,12
1808,2026,3453,Z,13
1809,2026,3158,Z,14
1810,2026,3239,Z,15


In [15]:
tourney = pd.read_csv('data_2026/WNCAATourneyCompactResults.csv')
tourney.columns = tourney.columns.str.upper()
tourney = tourney[['SEASON', 'WTEAMID', 'LTEAMID', 'WSCORE', 'LSCORE', 'DAYNUM']]
tourney

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,DAYNUM
0,1998,3104,3422,94,46,137
1,1998,3112,3365,75,63,137
2,1998,3163,3193,93,52,137
3,1998,3198,3266,59,45,137
4,1998,3203,3208,74,72,137
...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,147
1713,2025,3400,3395,58,47,147
1714,2025,3163,3417,85,51,151
1715,2025,3376,3400,74,57,151


In [16]:
conditions = [
    (tourney.DAYNUM >= 135) & (tourney.DAYNUM <= 136),
    (tourney.DAYNUM >= 137) & (tourney.DAYNUM <= 138),
    (tourney.DAYNUM >= 139) & (tourney.DAYNUM <= 140),
    (tourney.DAYNUM >= 144) & (tourney.DAYNUM <= 145),
    (tourney.DAYNUM >= 146) & (tourney.DAYNUM <= 147),
    tourney.DAYNUM == 151,
]
choices = [0, 1, 2, 3, 4, 5]
tourney_round = tourney.copy()
tourney_round['ROUND'] = np.select(conditions, choices, default=6)
tourney_round = tourney_round.drop(columns=['DAYNUM'])
tourney_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND
0,1998,3104,3422,94,46,1
1,1998,3112,3365,75,63,1
2,1998,3163,3193,93,52,1
3,1998,3198,3266,59,45,1
4,1998,3203,3208,74,72,1
...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,4
1713,2025,3400,3395,58,47,4
1714,2025,3163,3417,85,51,5
1715,2025,3376,3400,74,57,5


In [17]:
## Add in conference names, uppercase column headers and values and one hot encode
conf = pd.read_csv('data_2026/WTeamConferences.csv')
conf.columns = conf.columns.str.upper()
conf['CONFABBREV'] = conf['CONFABBREV'].str.upper().str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)
conf = conf.rename(columns={'SEASON': 'C_SEASON', 'TEAMID': 'C_TEAMID'})
conf

,C_SEASON,C_TEAMID,CONFABBREV
0,1998,3102,WAC
1,1998,3103,MAC
2,1998,3104,SEC
3,1998,3106,SWAC
4,1998,3108,SWAC
...,...,...,...
9848,2026,3477,SOUTHLAND
9849,2026,3478,NEC
9850,2026,3479,NEC
9851,2026,3480,A_SUN


In [18]:
tourney_conf_w = (
    tourney_round
    .merge(conf, left_on=['WTEAMID', 'SEASON'], right_on=['C_TEAMID', 'C_SEASON'])
    .drop(columns=['C_SEASON', 'C_TEAMID'])
    .rename(columns={'CONFABBREV': 'W_CONF'})
)
tourney_conf_w

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF
0,1998,3104,3422,94,46,1,SEC
1,1998,3112,3365,75,63,1,PAC_TEN
2,1998,3163,3193,93,52,1,BIG_EAST
3,1998,3198,3266,59,45,1,A_SUN
4,1998,3203,3208,74,72,1,A_TEN
...,...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,4,BIG_EAST
1713,2025,3400,3395,58,47,4,SEC
1714,2025,3163,3417,85,51,5,BIG_EAST
1715,2025,3376,3400,74,57,5,SEC


In [19]:
tourney_conf_round = (
    tourney_conf_w
    .merge(conf, left_on=['LTEAMID', 'SEASON'], right_on=['C_TEAMID', 'C_SEASON'])
    .drop(columns=['C_SEASON', 'C_TEAMID'])
    .rename(columns={'CONFABBREV': 'L_CONF'})
)
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF
0,1998,3104,3422,94,46,1,SEC,SOUTHERN
1,1998,3112,3365,75,63,1,PAC_TEN,WCC
2,1998,3163,3193,93,52,1,BIG_EAST,MAAC
3,1998,3198,3266,59,45,1,A_SUN,CUSA
4,1998,3203,3208,74,72,1,A_TEN,SEC
...,...,...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,4,BIG_EAST,BIG_TEN
1713,2025,3400,3395,58,47,4,SEC,BIG_TWELVE
1714,2025,3163,3417,85,51,5,BIG_EAST,BIG_TEN
1715,2025,3376,3400,74,57,5,SEC,SEC


In [20]:
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF
0,1998,3104,3422,94,46,1,SEC,SOUTHERN
1,1998,3112,3365,75,63,1,PAC_TEN,WCC
2,1998,3163,3193,93,52,1,BIG_EAST,MAAC
3,1998,3198,3266,59,45,1,A_SUN,CUSA
4,1998,3203,3208,74,72,1,A_TEN,SEC
...,...,...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,4,BIG_EAST,BIG_TEN
1713,2025,3400,3395,58,47,4,SEC,BIG_TWELVE
1714,2025,3163,3417,85,51,5,BIG_EAST,BIG_TEN
1715,2025,3376,3400,74,57,5,SEC,SEC


In [21]:
# Add winner seeds
w_t = (
    tourney_conf_round
    .merge(seed_value, left_on=['SEASON', 'WTEAMID'], right_on=['SEASON', 'TEAMID'])
    .drop(columns=['TEAMID'])
    .rename(columns={'REGION': 'W_REGION', 'SEED': 'W_SEED'})
)

# Add loser seeds
tourney_conf_round = (
    w_t
    .merge(seed_value, left_on=['SEASON', 'LTEAMID'], right_on=['SEASON', 'TEAMID'])
    .drop(columns=['TEAMID'])
    .rename(columns={'REGION': 'L_REGION', 'SEED': 'L_SEED'})
)

In [22]:
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF,W_REGION,W_SEED,L_REGION,L_SEED
0,1998,3104,3422,94,46,1,SEC,SOUTHERN,X,2,X,15
1,1998,3112,3365,75,63,1,PAC_TEN,WCC,W,3,W,14
2,1998,3163,3193,93,52,1,BIG_EAST,MAAC,W,2,W,15
3,1998,3198,3266,59,45,1,A_SUN,CUSA,Y,7,Y,10
4,1998,3203,3208,74,72,1,A_TEN,SEC,W,10,W,7
...,...,...,...,...,...,...,...,...,...,...,...,...
1712,2025,3163,3425,78,64,4,BIG_EAST,BIG_TEN,Z,2,Z,1
1713,2025,3400,3395,58,47,4,SEC,BIG_TWELVE,X,1,X,2
1714,2025,3163,3417,85,51,5,BIG_EAST,BIG_TEN,Z,2,Y,1
1715,2025,3376,3400,74,57,5,SEC,SEC,W,1,X,1


In [23]:
season_w = season.copy()
season_w.columns = ['W_' + c for c in season_w.columns]

season_l = season.copy()
season_l.columns = ['L_' + c for c in season_l.columns]

In [24]:
final = (
    tourney_conf_round
    .merge(
        season_w,
        left_on=['WTEAMID', 'SEASON'],
        right_on=['W_TEAMID', 'W_SEASON'],
    )
    .drop(columns=['W_TEAMID', 'W_SEASON'])
    .merge(
        season_l,
        left_on=['LTEAMID', 'SEASON'],
        right_on=['L_TEAMID', 'L_SEASON'],
    )
    .drop(columns=['L_TEAMID', 'L_SEASON'])
)
final

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF,W_REGION,W_SEED,...,L_PF_STDDEV,L_WLOCA,L_WLOCH,L_WLOCN,L_WINMARGINMEDIAN,L_WINMARGINMEAN,L_LOSEMARGINMEDIAN,L_LOSEMARGINMEAN,L_WON_CONFERENCE,L_TOTAL_WINS
0,2010,3124,3201,69,55,1,BIG_TWELVE,WAC,X,4,...,4.084570,11.0,13.0,3.0,12.0,18.259259,9.5,11.333333,0.0,27.0
1,2010,3173,3395,67,66,1,A_TEN,MWC,X,8,...,3.400642,6.0,15.0,1.0,18.0,19.818182,6.5,9.500000,0.0,22.0
2,2010,3181,3214,72,37,1,ACC,MEAC,X,2,...,5.350368,5.0,10.0,4.0,17.0,17.157895,9.0,8.636364,1.0,19.0
3,2010,3199,3256,75,61,1,ACC,WAC,W,3,...,4.757337,11.0,9.0,3.0,12.0,15.782609,6.5,6.875000,1.0,23.0
4,2010,3207,3265,62,42,1,BIG_EAST,MAAC,X,5,...,3.137324,7.0,18.0,1.0,14.0,14.884615,5.0,6.857143,1.0,26.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,2025,3163,3425,78,64,4,BIG_EAST,BIG_TEN,Z,2,...,3.913308,9.0,14.0,5.0,20.0,24.821429,7.0,8.333333,0.0,28.0
905,2025,3400,3395,58,47,4,SEC,BIG_TWELVE,X,1,...,2.996135,7.0,19.0,5.0,21.0,23.645161,9.0,14.333333,1.0,31.0
906,2025,3163,3417,85,51,5,BIG_EAST,BIG_TEN,Z,2,...,4.136653,10.0,12.0,8.0,17.0,23.166667,12.0,12.000000,1.0,30.0
907,2025,3376,3400,74,57,5,SEC,SEC,W,1,...,4.643436,10.0,15.0,6.0,26.0,26.741935,17.0,15.333333,0.0,31.0


In [25]:
final = pd.get_dummies(final, columns=['W_CONF', 'L_CONF'], drop_first=True, dtype=int)

In [26]:
wloc_cols = ['W_WLOCN', 'W_WLOCH', 'W_WLOCA', 'L_WLOCN', 'L_WLOCH', 'L_WLOCA']
final[wloc_cols] = final[wloc_cols].astype(int)

### This table is all season data joined with historic tournament data

In [27]:
final.to_csv('data_2026/final_features_W.csv', index=False)

### Create season table for predicting 2026

In [28]:
conf_simple = pd.read_csv('data_2026/WTeamConferences.csv')
conf_simple.columns = conf_simple.columns.str.upper()
conf_simple['CONFABBREV'] = conf_simple['CONFABBREV'].str.upper().str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)

season = season.merge(
    conf_simple[['SEASON', 'TEAMID', 'CONFABBREV']],
    on=['SEASON', 'TEAMID'],
).rename(columns={'CONFABBREV': 'CONF'})

season = pd.get_dummies(season, columns=['CONF'], drop_first=True, dtype=int)

In [29]:
# Save season stats for all teams (not filtered by seeds) so future seasons are included
season.to_csv('data_2026/final_season_stats_W.csv', index=False)